# run_accuracy.ipynb

Evaluate **base model accuracy** on DATA JSONL using the **all_at_once** prompt.

What this notebook does:
1. Auto-detect repo root.
2. Load DATA JSONL (no prompts inside).
3. Load `system.txt` and `user_template.txt` from `02_prompts/all_at_once/`.
4. Build messages for each example.
5. Call a model (OpenAI by default; pluggable).
6. Parse strict JSON outputs.
7. Compute accuracy + precision/recall/F1 per label.
8. Save a run folder under:
   `datasets/<dataset_id>/03_accuracy_testing/all_at_once/<provider>/<model>/runs/<timestamp>/`

You must set your API key in the environment (recommended) or in the notebook.


## 0)  Imports & API Key Setup

In [2]:
# --- 0) Imports ---
import os, json, time, re
from pathlib import Path

import json, time
from pathlib import Path

# Find repo root
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
    raise RuntimeError(f"Repo root not found from start={start}")

repo_root = find_repo_root()
print("Repo root:", repo_root)

# Load API key
KEYS_PATH = repo_root / "keys.json"
if not KEYS_PATH.exists():
    raise FileNotFoundError(f"keys.json missing: {KEYS_PATH}")

keys = json.loads(KEYS_PATH.read_text())
OPENAI_API_KEY = keys.get("openai")
if not OPENAI_API_KEY:
    raise ValueError("Missing 'openai' key in keys.json")

# Install + init OpenAI
%pip install -q openai
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client initialized.")


Repo root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
OpenAI client initialized.


## 1) Paths & Model Selection

In [3]:
from pathlib import Path
from datetime import datetime, timezone
import re, json

# ------------------------------------------------------------------
# Dataset root
# ------------------------------------------------------------------
dataset_root = repo_root / "datasets" / "yt_factlink"
print("Dataset root:", dataset_root)

# ------------------------------------------------------------------
# Dataset ID (used in config metadata)
# ------------------------------------------------------------------

DATASET_ID = "yt_factlink"

# ------------------------------------------------------------------
# Choose evaluation source
# ------------------------------------------------------------------
USE_SPLIT = (
    True  # True = evaluate on test split, False = evaluate on full canonical file
)

SPLIT_NAME = "split_v1_seed42"  # used only if USE_SPLIT=True
# ------------------------------------------------------------------
# Choose provider + model to evaluate
# ------------------------------------------------------------------
PROVIDER = "openai"
MODEL_NAME = "gpt-4.1-2025-04-14"  # model you want to TEST
SUFFIX_TAG = "single_class" # experiment name (prompt set)

# ------------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------------
if USE_SPLIT:
    SPLIT_DIR = dataset_root / "01_conversion" / "03_splits" / SPLIT_NAME
    EVAL_JSONL = SPLIT_DIR / "test.data.jsonl"

    assert SPLIT_DIR.exists(), f"Missing split folder: {SPLIT_DIR}"
    assert EVAL_JSONL.exists(), f"Missing test split: {EVAL_JSONL}"

    print("Evaluation mode: TEST SPLIT")
    print("Using split:", SPLIT_NAME)
else:
    EVAL_JSONL = (
        dataset_root
        / "01_conversion"
        / "02_outputs"
        / "manual_labels_386_v2.data.jsonl"
    )
    assert EVAL_JSONL.exists(), f"Canonical data JSONL missing: {EVAL_JSONL}"

    print("Evaluation mode: FULL CANONICAL (no split)")
    print("Using full dataset:", EVAL_JSONL)

print("Eval file:", EVAL_JSONL)


# ------------------------------------------------------------------
# Prompt paths (per-class prompts under single_class/C1, C2, ...)
# ------------------------------------------------------------------
LABEL_KEYS = ["C1", "C2", "C3", "C4", "C6"]

PROMPT_ROOT = dataset_root / "02_prompts" / SUFFIX_TAG
assert PROMPT_ROOT.exists(), f"Missing prompt root: {PROMPT_ROOT}"

print("Prompt root:", PROMPT_ROOT)

# Dict: { "C1": {"system": "...", "user": "..."}, ... }
class_prompts: dict[str, dict[str, str]] = {}

for key in LABEL_KEYS:
    class_dir = PROMPT_ROOT / key
    system_path = class_dir / "system.txt"
    user_path = class_dir / "user.txt"

    assert class_dir.exists(), f"Missing class prompt folder: {class_dir}"
    assert system_path.exists(), f"Missing system prompt: {system_path}"
    assert user_path.exists(), f"Missing user prompt: {user_path}"

    class_prompts[key] = {
        "system": system_path.read_text(encoding="utf-8"),
        "user": user_path.read_text(encoding="utf-8"),
    }

print("Loaded prompts for classes:", ", ".join(class_prompts.keys()))

# ------------------------------------------------------------------
# Helper: slugify for safe folder names
# ------------------------------------------------------------------
def slugify(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9._-]+", "-", s)
    s = re.sub(r"-{2,}", "-", s)
    return s.strip("-")


# ------------------------------------------------------------------
# Accuracy run folders (mirrors finetuning style)
# Notebook lives inside provider folder (e.g., openai/)
# Structure:
#   <NOTEBOOK_DIR>/<MODEL_NAME>/runs/<run_name>/
# ------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()

MODEL_DIR = NOTEBOOK_DIR / slugify(MODEL_NAME)
MODEL_DIR.mkdir(exist_ok=True)
(MODEL_DIR / ".gitkeep").write_text("")

RUNS_ROOT = MODEL_DIR / "runs"
RUNS_ROOT.mkdir(exist_ok=True)
(RUNS_ROOT / ".gitkeep").write_text("")

timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")

run_name = "_".join(
    [
        timestamp,
        slugify(PROVIDER),
        slugify(MODEL_NAME),
        slugify(SUFFIX_TAG),
        slugify(SPLIT_NAME),
    ]
)

RUN_DIR = RUNS_ROOT / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("Notebook directory:", NOTEBOOK_DIR.resolve())
print("Model directory   :", MODEL_DIR.resolve())
print("Runs root         :", RUNS_ROOT.resolve())
print("Run directory     :", RUN_DIR.resolve())

# ------------------------------------------------------------------
# Create run subfolders early (so later cells can write into them)
# ------------------------------------------------------------------
INPUTS_DIR = RUN_DIR / "01_inputs"
OUTPUTS_DIR = RUN_DIR / "02_outputs"
SNAPSHOTS_DIR = RUN_DIR / "03_snapshots"

INPUTS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)
SNAPSHOTS_DIR.mkdir(exist_ok=True)

(INPUTS_DIR / ".gitkeep").write_text("")
(OUTPUTS_DIR / ".gitkeep").write_text("")
(SNAPSHOTS_DIR / ".gitkeep").write_text("")

# Save per-class prompt snapshots into inputs for reproducibility
for key, prompts in class_prompts.items():
    class_input_dir = INPUTS_DIR / key
    class_input_dir.mkdir(exist_ok=True)

    (class_input_dir / "system.txt").write_text(prompts["system"], encoding="utf-8")
    (class_input_dir / "user.txt").write_text(prompts["user"], encoding="utf-8")

# Save only the relative path of the eval dataset (no copying)
relative_eval_path = str(EVAL_JSONL.relative_to(repo_root))

(INPUTS_DIR / "eval_file.txt").write_text(relative_eval_path, encoding="utf-8")

print("Saved eval file reference:", relative_eval_path)

print("\n=== Configuration OK ===")
print("Provider   :", PROVIDER)
print("Model      :", MODEL_NAME)
print("Prompt tag :", SUFFIX_TAG)
print("Split      :", SPLIT_NAME)
print("Run dir    :", RUN_DIR)

Dataset root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink
Evaluation mode: TEST SPLIT
Using split: split_v1_seed42
Eval file: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/03_splits/split_v1_seed42/test.data.jsonl
Prompt root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/02_prompts/single_class
Loaded prompts for classes: C1, C2, C3, C4, C6
Notebook directory: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai
Model directory   : /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAna

## 3) Load Eval Data

In [4]:
def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test_rows = read_jsonl(EVAL_JSONL)
print("Test examples:", len(test_rows))


Test examples: 77


## 4) Build user message from placeholders

In [5]:
def build_user_message(user_template: str, row: dict) -> str:
    return (
        user_template
        .replace("[VIDEOTITLE]", row.get("VIDEOTITEL", ""))
        .replace("[VIDEOCONTEXT]", row.get("VIDEOCONTEXT", ""))
        .replace("[TARGETCOMMENT]", row.get("TARGETCOMMENT", ""))
    )


## 5) Model client (OpenAI default)

If you're not using OpenAI, replace this section with your provider's SDK.


In [6]:
import json

# --- OpenAI Client Setup ---
%pip install openai
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

#-- Define Structured Output Model ---
from typing import List
from pydantic import BaseModel

class SingleLabelOutput(BaseModel):
    value: int  # must be 0 or 1

#--- Function to Call Model ---
def call_model(
    label_key: str,
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.0,
) -> tuple[int, str]:
    """
    Call the LLM for a single label (C1, C2, ...) and return (value, raw_json).
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    resp = client.responses.parse(
        model=MODEL_NAME,
        input=messages,
        text_format=SingleLabelOutput,  # single-label schema
        temperature=temperature,
    )

    parsed = resp.output_parsed  # SingleLabelOutput
    v = parsed.value
    raw_json = parsed.model_dump_json()

    if v not in (0, 1):
        raise ValueError(f"Model for {label_key} returned invalid value: {v}")

    return v, raw_json



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 6) Run inference

In [7]:
from IPython.display import clear_output

LABEL_KEYS = ["C1","C2","C3","C4","C6"]  # adjust if you add more

predictions = []
failures = 0

for i, row in enumerate(test_rows):
    row_pred: dict[str, int] = {}
    row_raw: dict[str, str] = {}
    
    print(f"\n====================")
    print(f"Processing row {i+1}/{len(test_rows)}")
    print("Comment:", row.get("TARGETCOMMENT"))
    print("====================\n")

    for key in LABEL_KEYS:
        prompts = class_prompts[key]  # {"system": ..., "user": ...}
        system_prompt = prompts["system"]
        user_template = prompts["user"]
        user_msg = build_user_message(user_template, row)

        print(f" → Running label {key}...")

        try:
            value, raw_json = call_model(
                label_key=key,
                system_prompt=system_prompt,
                user_prompt=user_msg,
                temperature=0.0,
            )
            row_pred[key] = int(value)
            row_raw[key] = raw_json
            
            print(f"   ✓ {key} = {value}")

        except Exception as e:
            failures += 1
            row_pred[key] = 0
            row_raw[key] = f"<PARSE FAILURE: {e}>"
            
            print(f"   ✗ ERROR for {key}: {e}")


    predictions.append({
        "pred": row_pred,
        "gold": row["labels"],   # assuming this is still a dict {"C1": ..., ...}
        "raw_output": row_raw,   # now per-label raw JSON
    })

    clear_output(wait=True)
    print(f"[{i+1}/{len(test_rows)}]")
    print(row.get("TARGETCOMMENT"))
    print("GOLD:", row["labels"])
    print("PRED:", row_pred)

print("Done. Total failures:", failures)
predictions[0]


[77/77]
台灣的交通罰單是隨車。日本是隨人，照相舉發須能看清駕駛臉，才能開罰。
GOLD: {'C1': 0, 'C2': 0, 'C3': 0, 'C4': 0, 'C6': 0}
PRED: {'C1': 0, 'C2': 0, 'C3': 0, 'C4': 0, 'C6': 0}
Done. Total failures: 0


{'pred': {'C1': 1, 'C2': 1, 'C3': 0, 'C4': 0, 'C6': 0},
 'gold': {'C1': 1, 'C2': 1, 'C3': 0, 'C4': 0, 'C6': 0},
 'raw_output': {'C1': '{"value":1}',
  'C2': '{"value":1}',
  'C3': '{"value":0}',
  'C4': '{"value":0}',
  'C6': '{"value":0}'}}

## 7) Compute metrics

In [10]:
def compute_metrics(predictions):
    # per-label confusion counts
    counts = {k: {"tp":0,"fp":0,"tn":0,"fn":0} for k in LABEL_KEYS}

    for ex in predictions:
        gold = ex["gold"]
        pred = ex["pred"]
        for k in LABEL_KEYS:
            g = int(gold.get(k,0))
            p = int(pred.get(k,0))
            if p==1 and g==1: counts[k]["tp"] += 1
            elif p==1 and g==0: counts[k]["fp"] += 1
            elif p==0 and g==0: counts[k]["tn"] += 1
            elif p==0 and g==1: counts[k]["fn"] += 1

    metrics = {}
    for k,c in counts.items():
        tp,fp,tn,fn = c["tp"],c["fp"],c["tn"],c["fn"]
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1   = (2*prec*rec/(prec+rec)) if (prec+rec)>0 else 0.0
        acc  = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn)>0 else 0.0
        metrics[k] = {"precision":prec, "recall":rec, "f1":f1, "accuracy":acc, **c}

    # macro averages
    macro_f1 = sum(metrics[k]["f1"] for k in LABEL_KEYS)/len(LABEL_KEYS)
    macro_acc = sum(metrics[k]["accuracy"] for k in LABEL_KEYS)/len(LABEL_KEYS)

    return metrics, {"macro_f1": macro_f1, "macro_accuracy": macro_acc}

per_label, overall = compute_metrics(predictions)
per_label, overall


({'C1': {'precision': 0.8285714285714286,
   'recall': 0.7837837837837838,
   'f1': 0.8055555555555555,
   'accuracy': 0.8181818181818182,
   'tp': 29,
   'fp': 6,
   'tn': 34,
   'fn': 8},
  'C2': {'precision': 0.6666666666666666,
   'recall': 0.9411764705882353,
   'f1': 0.7804878048780487,
   'accuracy': 0.8831168831168831,
   'tp': 16,
   'fp': 8,
   'tn': 52,
   'fn': 1},
  'C3': {'precision': 0.3333333333333333,
   'recall': 0.2857142857142857,
   'f1': 0.30769230769230765,
   'accuracy': 0.8831168831168831,
   'tp': 2,
   'fp': 4,
   'tn': 66,
   'fn': 5},
  'C4': {'precision': 0.42857142857142855,
   'recall': 0.75,
   'f1': 0.5454545454545454,
   'accuracy': 0.935064935064935,
   'tp': 3,
   'fp': 4,
   'tn': 69,
   'fn': 1},
  'C6': {'precision': 0.0,
   'recall': 0.0,
   'f1': 0.0,
   'accuracy': 1.0,
   'tp': 0,
   'fp': 0,
   'tn': 77,
   'fn': 0}},
 {'macro_f1': 0.48783804271609144, 'macro_accuracy': 0.903896103896104})

## 8) Save run artifacts

In [ ]:
import json
import shutil
from pathlib import Path

# -----------------------------
# Save run config into OUTPUTS_DIR
# -----------------------------
run_config = {
    "dataset_id": DATASET_ID,
    "experiment_type": SUFFIX_TAG,
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "data_jsonl": str(EVAL_JSONL.relative_to(repo_root)),
    "prompt_dir": str(PROMPT_ROOT.relative_to(repo_root)),
    "timestamp": timestamp,
}

(OUTPUTS_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# -----------------------------
# Save predictions into OUTPUTS_DIR
# -----------------------------
with (OUTPUTS_DIR / "preds.jsonl").open("w", encoding="utf-8") as f:
    for ex in predictions:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

# -----------------------------
# Save full metrics into OUTPUTS_DIR
# -----------------------------
metrics_payload = {
    "failures": failures,
    "overall": overall,
    "per_label": per_label
}

(OUTPUTS_DIR / "metrics.json").write_text(
    json.dumps(metrics_payload, indent=2, ensure_ascii=False),
    encoding="utf-8"
)


# -----------------------------
# Snapshot THIS notebook into SNAPSHOTS_DIR
# -----------------------------
NOTEBOOK_FILENAME = "run_accuracy.ipynb"  # change if your notebook has a different name
NOTEBOOK_PATH = Path(NOTEBOOK_FILENAME)

if NOTEBOOK_PATH.exists():
    shutil.copy2(NOTEBOOK_PATH, SNAPSHOTS_DIR / NOTEBOOK_FILENAME)
    print("Notebook snapshot saved to:", (SNAPSHOTS_DIR / NOTEBOOK_FILENAME).resolve())
else:
    print(f"WARNING: {NOTEBOOK_FILENAME} not found in CWD. "
          "Start Jupyter from the notebook folder or update NOTEBOOK_FILENAME.")

print("Saved run to outputs:", OUTPUTS_DIR.resolve())
print("Snapshots in:", SNAPSHOTS_DIR.resolve())



Notebook snapshot saved to: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1-2025-04-14/runs/2025-11-28T115707Z_openai_gpt-4.1-2025-04-14_single_class_split_v1_seed42/03_snapshots/run_accuracy.ipynb
Saved run to outputs: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1-2025-04-14/runs/2025-11-28T115707Z_openai_gpt-4.1-2025-04-14_single_class_split_v1_seed42/02_outputs
Snapshots in: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_accuracy_testing/single_class/openai/gpt-4.1-2025-04-14/runs/2025-11-28T115707Z_openai_gpt-4.1-2025-04-14_single_class_split_v1_seed42/03_snapshots
